In [64]:
from __future__ import annotations

import os
import re
import json
import math
import dataclasses
import hashlib
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

# ===================== CONFIG =====================
INPUT_JSONL = "dataoutput_dataset.jsonl"  
OUT_DIR = "qnn_dataset/qnn_dataset_v2"

# Chế độ dataset:
# - "all": giữ mọi version (fixed, vulnerable, vulnerable_other)
# - "clean_binary": chỉ giữ fixed vs vulnerable (nhãn sạch hơn)
MODE = "all"

# Feature params
DIM = 128
SEED = 42
MAX_NODES = 4000
MAX_PATH_LEN = 8

# scalar dims (khuyến nghị 10 để có parse_ok + size_total)
SCALAR_DIMS = 10

USE_EDGES = True
USE_PATHS = True

# Ổn định scale cho count-features
CLIP_COUNTS = 50.0
APPLY_LOG1P = True

os.makedirs(OUT_DIR, exist_ok=True)

print("✓ Config loaded")
print("  INPUT_JSONL:", INPUT_JSONL)
print("  OUT_DIR    :", OUT_DIR)
print("  MODE       :", MODE)


✓ Config loaded
  INPUT_JSONL: dataoutput_dataset.jsonl
  OUT_DIR    : qnn_dataset/qnn_dataset_v2
  MODE       : all


Utils: đọc JSONL, chuẩn hoá code, hashing ổn định

In [65]:
def iter_jsonl(path: str) -> Iterable[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON decode error at line {line_no}: {e}") from e

def normalize_newlines(code: str) -> str:
    return code.replace("\r\n", "\n").replace("\r", "\n")

def strip_java_comments(code: str) -> str:
    # đủ dùng để giảm parse lỗi do comment; không phải parser hoàn hảo 100%
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.S)
    code = re.sub(r"//.*?$", "", code, flags=re.M)
    return code

def stable_hash(text: str, seed: int = 0) -> int:
    h = hashlib.md5((str(seed) + "::" + text).encode("utf-8")).hexdigest()
    return int(h, 16)

def hash_bucket(text: str, dim: int, seed: int = 0) -> int:
    return stable_hash(text, seed) % dim

def hash_sign(text: str, seed: int = 0) -> float:
    # signed hashing (±1) để giảm bias collision
    return 1.0 if (stable_hash("sign::" + text, seed) & 1) == 0 else -1.0

print("Utils ready")


Utils ready


Tree-sitter Java parser (wrap snippet để parse ổn)

In [66]:
# Cài đặt thư viện tree-sitter đúng phiên bản
import sys
import subprocess

print("Đang cài đặt tree-sitter...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "tree-sitter==0.21.3",
    "tree-sitter-java==0.21.0",
    "-q"
])
print("✓ Hoàn tất!")

Đang cài đặt tree-sitter...
✓ Hoàn tất!


In [67]:
class JavaASTParser:
    def __init__(self):
        import tree_sitter
        
        # Thử nhiều phương pháp khác nhau
        parser_created = False
        
        # Phương pháp 1: tree_sitter_languages với get_parser (API mới nhất)
        try:
            from tree_sitter_languages import get_parser
            self._parser = get_parser("java")
            parser_created = True
        except:
            pass
        
        # Phương pháp 2: tree_sitter_java với API mới (Language 1 param)
        if not parser_created:
            try:
                import tree_sitter_java
                java_lang = tree_sitter.Language(tree_sitter_java.language())
                self._parser = tree_sitter.Parser(java_lang)
                parser_created = True
            except:
                pass
        
        # Phương pháp 3: tree_sitter_java với API cũ (set_language)
        if not parser_created:
            try:
                import tree_sitter_java
                java_lang = tree_sitter.Language(tree_sitter_java.language())
                self._parser = tree_sitter.Parser()
                self._parser.language = java_lang
                parser_created = True
            except:
                pass
        
        if not parser_created:
            raise ImportError(
                "Không thể khởi tạo Java parser.\n"
                "Hãy thử cài đặt lại:\n"
                "  pip uninstall -y tree-sitter tree-sitter-languages tree-sitter-java\n"
                "  pip install tree-sitter==0.21.3 tree-sitter-java==0.21.0"
            )

    @staticmethod
    def _wrap_as_class(snippet: str) -> str:
        return "class Dummy {\n" + snippet + "\n}\n"

    @staticmethod
    def _wrap_as_method_body(snippet: str) -> str:
        return "class Dummy {\nvoid __dummy__(){\n" + snippet + "\n}\n}\n"

    @staticmethod
    def _count_error_nodes(root) -> int:
        cnt = 0
        stack = [root]
        while stack:
            n = stack.pop()
            if n.type == "ERROR":
                cnt += 1
            stack.extend(n.children)
        return cnt

    def parse_best_effort(self, snippet: str):
        # chiến lược 1: wrap class
        src = self._wrap_as_class(snippet)
        tree = self._parser.parse(bytes(src, "utf-8"))
        root = tree.root_node
        err1 = self._count_error_nodes(root)

        # chiến lược 2: wrap method body nếu lỗi nhiều
        if err1 > 0:
            src2 = self._wrap_as_method_body(snippet)
            tree2 = self._parser.parse(bytes(src2, "utf-8"))
            root2 = tree2.root_node
            err2 = self._count_error_nodes(root2)
            if err2 < err1:
                return root2, err2

        return root, err1

print("✓ Tree-sitter parser ready")

✓ Tree-sitter parser ready


Feature extractor “QNN-ready” (AST → vector cố định)

In [68]:
@dataclasses.dataclass
class ASTStats:
    n_nodes: int = 0
    n_edges: int = 0
    n_leaves: int = 0
    max_depth: int = 0
    avg_branching: float = 0.0
    truncated: int = 0
    error_nodes: int = 0
    parse_ok: int = 1

class ASTHasherFeaturizerV2:
    """
    Fixes so với bản cũ:
    (1) tách bucket theo nhóm node/edge/path để giảm collision giữa nhóm
    (2) signed hashing
    (3) clip + log1p + L2-normalize phần hashed để ổn định scale trước QNN
    (4) parse_fail không dùng 999999 (tránh phá scale), có parse_ok=0
    """
    def __init__(
        self,
        dim: int = 128,
        seed: int = 42,
        max_nodes: int = 4000,
        max_path_len: int = 8,
        scalar_dims: int = 10,
        use_edges: bool = True,
        use_paths: bool = True,
        clip_counts: float = 50.0,
        apply_log1p: bool = True,
    ):
        if dim <= scalar_dims:
            raise ValueError("dim phải > scalar_dims")
        self.dim = dim
        self.seed = seed
        self.max_nodes = max_nodes
        self.max_path_len = max_path_len
        self.scalar_dims = scalar_dims
        self.use_edges = use_edges
        self.use_paths = use_paths
        self.clip_counts = clip_counts
        self.apply_log1p = apply_log1p

        self.hashed_dim = dim - scalar_dims

        # chia hashed_dim thành 3 vùng: node/edge/path
        self.node_dim = self.hashed_dim // 3
        self.edge_dim = self.hashed_dim // 3
        self.path_dim = self.hashed_dim - self.node_dim - self.edge_dim

        self._parser = JavaASTParser()

    def featurize(self, code: str) -> Tuple[np.ndarray, ASTStats]:
        code = normalize_newlines(strip_java_comments(code))
        x = np.zeros((self.dim,), dtype=np.float32)

        try:
            root, err_nodes = self._parser.parse_best_effort(code)

            node_types: List[str] = []
            parent: List[int] = []
            depth: List[int] = []
            child_count: List[int] = []

            stack = [(root, -1, 0)]
            truncated = 0

            while stack:
                n, p, d = stack.pop()
                if len(node_types) >= self.max_nodes:
                    truncated = 1
                    break

                nid = len(node_types)
                node_types.append(n.type)
                parent.append(p)
                depth.append(d)
                child_count.append(len(n.children))

                for ch in reversed(n.children):
                    stack.append((ch, nid, d + 1))

            n_nodes = len(node_types)
            n_edges = sum(1 for p in parent if p != -1)
            n_leaves = sum(1 for c in child_count[:n_nodes] if c == 0)
            max_depth = max(depth) if depth else 0
            avg_branching = float(sum(child_count[:n_nodes]) / max(1, n_nodes))

            stats = ASTStats(
                n_nodes=n_nodes,
                n_edges=n_edges,
                n_leaves=n_leaves,
                max_depth=max_depth,
                avg_branching=avg_branching,
                truncated=truncated,
                error_nodes=err_nodes,
                parse_ok=1,
            )

            # -------- A) Node buckets --------
            for t in node_types:
                k = "NT:" + t
                b = hash_bucket(k, self.node_dim, self.seed)
                x[b] += hash_sign(k, self.seed)

            # -------- B) Edge buckets --------
            if self.use_edges:
                base = self.node_dim
                for i in range(n_nodes):
                    p = parent[i]
                    if p == -1:
                        continue
                    k = f"E:{node_types[p]}->{node_types[i]}"
                    b = hash_bucket(k, self.edge_dim, self.seed) + base
                    x[b] += hash_sign(k, self.seed)

            # -------- C) Path buckets --------
            if self.use_paths and n_nodes > 0:
                base = self.node_dim + self.edge_dim
                for leaf_id in range(n_nodes):
                    if child_count[leaf_id] != 0:
                        continue
                    path_types = []
                    cur = leaf_id
                    steps = 0
                    while cur != -1 and steps < self.max_path_len:
                        path_types.append(node_types[cur])
                        cur = parent[cur]
                        steps += 1
                    path_types.reverse()
                    if not path_types:
                        continue
                    k = "P:" + "/".join(path_types)
                    b = hash_bucket(k, self.path_dim, self.seed) + base
                    x[b] += hash_sign(k, self.seed)

            # -------- stabilize hashed part --------
            hashed = x[:self.hashed_dim]
            hashed = np.clip(hashed, -self.clip_counts, self.clip_counts)
            if self.apply_log1p:
                hashed = np.sign(hashed) * np.log1p(np.abs(hashed))

            # L2 normalize: giảm lệ thuộc kích thước AST (rất quan trọng cho QNN)
            norm = float(np.linalg.norm(hashed) + 1e-8)
            hashed = hashed / norm
            x[:self.hashed_dim] = hashed.astype(np.float32)

            # -------- scalar stats --------
            size_total = float(n_nodes + n_edges + n_leaves)
            scalars = [
                math.log1p(n_nodes),
                math.log1p(n_edges),
                math.log1p(n_leaves),
                math.log1p(max_depth),
                float(avg_branching),
                float(truncated),
                math.log1p(min(err_nodes, 50)),   # clip lỗi parse
                float(stats.parse_ok),
                math.log1p(size_total),
                1.0,                              # bias
            ]
            scalars = (scalars + [0.0] * self.scalar_dims)[: self.scalar_dims]
            x[self.hashed_dim:] = np.asarray(scalars, dtype=np.float32)

            return x, stats

        except Exception:
            # parse fail: không dùng số cực lớn; gắn parse_ok=0
            stats = ASTStats(truncated=1, error_nodes=0, parse_ok=0)
            # scalars tối thiểu (và bias = 1)
            scalars = [0,0,0,0,0,1,0,0,0,1][: self.scalar_dims]
            x[self.hashed_dim:] = np.asarray(scalars, dtype=np.float32)
            return x, stats

print("Featurizer V2 ready")


Featurizer V2 ready


Load JSONL + thống kê cơ bản + lọc theo MODE

In [69]:
print("Loading JSONL...")
raw_records = list(iter_jsonl(INPUT_JSONL))
print("  total records:", len(raw_records))

df_raw = pd.DataFrame(raw_records)

print("\nColumns:", df_raw.columns.tolist())
print("\nLabel distribution:")
print(df_raw["label"].value_counts(dropna=False))

if "version" in df_raw.columns:
    print("\nVersion distribution:")
    print(df_raw["version"].value_counts(dropna=False))

if "vul_id" in df_raw.columns:
    print("\nUnique vul_id:", df_raw["vul_id"].nunique())

# MODE handling
if MODE == "clean_binary":
    df_raw = df_raw[df_raw["version"].isin(["fixed", "vulnerable"])].copy()
    print("\n✓ MODE clean_binary applied")
    print("  remaining:", len(df_raw))
elif MODE == "all":
    print("\n✓ MODE all (no filtering)")
else:
    raise ValueError("MODE must be 'all' or 'clean_binary'")


Loading JSONL...
  total records: 1865

Columns: ['vul_id', 'file', 'method', 'version', 'label', 'code']

Label distribution:
label
0    1426
1     439
Name: count, dtype: int64

Version distribution:
version
vulnerable_other    992
vulnerable          439
fixed               434
Name: count, dtype: int64

Unique vul_id: 81

✓ MODE all (no filtering)


Dedup (giảm bias do snippet lặp) + tạo `id`

In [70]:
def make_row_id(r: pd.Series) -> str:
    return f'{r.get("vul_id","")}::{r.get("file","")}::{r.get("method","")}::{r.get("version","")}'

def dedup_df(df: pd.DataFrame) -> pd.DataFrame:
    keys = []
    for _, r in df.iterrows():
        code = normalize_newlines(str(r.get("code", "")))
        key = hashlib.md5(
            (str(r.get("vul_id","")) + "\n" +
             str(r.get("file","")) + "\n" +
             str(r.get("method","")) + "\n" +
             str(r.get("version","")) + "\n" +
             str(r.get("label",0)) + "\n" +
             code).encode("utf-8")
        ).hexdigest()
        keys.append(key)
    df2 = df.copy()
    df2["_dedup_key"] = keys
    before = len(df2)
    df2 = df2.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])
    after = len(df2)
    print(f"Dedup: {before} -> {after} (removed {before-after})")
    return df2

df = dedup_df(df_raw)
df["id"] = df.apply(make_row_id, axis=1)

# ensure types
df["label"] = df["label"].astype(int)
print("✓ Dedup + id ready. rows:", len(df))
df[["id", "vul_id", "version", "label"]].head(5)


Dedup: 1865 -> 1578 (removed 287)
✓ Dedup + id ready. rows: 1578


,id,vul_id,version,label
0,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,VUL4J-1,vulnerable,1
1,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,VUL4J-1,fixed,0
2,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,VUL4J-1,vulnerable_other,0
3,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,VUL4J-1,vulnerable_other,0
4,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,VUL4J-1,vulnerable_other,0


Feature extraction (JSONL → vectors) + log parse fail

In [71]:
featurizer = ASTHasherFeaturizerV2(
    dim=DIM,
    seed=SEED,
    max_nodes=MAX_NODES,
    max_path_len=MAX_PATH_LEN,
    scalar_dims=SCALAR_DIMS,
    use_edges=USE_EDGES,
    use_paths=USE_PATHS,
    clip_counts=CLIP_COUNTS,
    apply_log1p=APPLY_LOG1P,
)

failed_path = os.path.join(OUT_DIR, "failed_parse.jsonl")
out_rows = []

n = len(df)
n_fail = 0
n_trunc = 0

with open(failed_path, "w", encoding="utf-8") as f_fail:
    for i, r in df.iterrows():
        if (len(out_rows) + 1) % 200 == 0:
            print(f"  processed {len(out_rows)+1}/{n}")

        code = str(r.get("code", ""))
        vec, st = featurizer.featurize(code)

        if st.parse_ok == 0:
            n_fail += 1
            f_fail.write(json.dumps({
                "id": r["id"],
                "vul_id": r.get("vul_id",""),
                "file": r.get("file",""),
                "method": r.get("method",""),
                "version": r.get("version",""),
                "label": int(r.get("label", 0)),
                "note": "parse_ok=0",
                "code_head": normalize_newlines(code)[:4000],
            }, ensure_ascii=False) + "\n")

        if st.truncated == 1:
            n_trunc += 1

        row = {
            "id": r["id"],
            "vul_id": r.get("vul_id",""),
            "file": r.get("file",""),
            "method": r.get("method",""),
            "version": r.get("version",""),
            "label": int(r.get("label", 0)),

            "ast_n_nodes": st.n_nodes,
            "ast_n_edges": st.n_edges,
            "ast_n_leaves": st.n_leaves,
            "ast_max_depth": st.max_depth,
            "ast_avg_branching": st.avg_branching,
            "ast_error_nodes": st.error_nodes,
            "ast_truncated": st.truncated,
            "parse_ok": st.parse_ok,
        }

        for j in range(DIM):
            row[f"feat_{j}"] = float(vec[j])

        out_rows.append(row)

print("✓ Feature extraction done")
print("  parse_fail :", n_fail)
print("  truncated  :", n_trunc)
print("  failed_log :", failed_path)

  processed 200/1578
  processed 400/1578
  processed 600/1578
  processed 800/1578
  processed 1000/1578
  processed 1200/1578
  processed 1400/1578
✓ Feature extraction done
  parse_fail : 0
  truncated  : 2
  failed_log : qnn_dataset/qnn_dataset_v2\failed_parse.jsonl


In [72]:
df_out = pd.DataFrame(out_rows)

csv_path = os.path.join(OUT_DIR, "qnn_features_v2.csv")
df_out.to_csv(csv_path, index=False)

summary = {
    "input_jsonl": os.path.abspath(INPUT_JSONL),
    "output_csv": os.path.abspath(csv_path),
    "out_dir": os.path.abspath(OUT_DIR),
    "mode": MODE,
    "feature_config": {
        "dim": DIM,
        "seed": SEED,
        "max_nodes": MAX_NODES,
        "max_path_len": MAX_PATH_LEN,
        "scalar_dims": SCALAR_DIMS,
        "use_edges": USE_EDGES,
        "use_paths": USE_PATHS,
        "clip_counts": CLIP_COUNTS,
        "apply_log1p": APPLY_LOG1P,
    },
    "dataset_stats": {
        "n": int(len(df_out)),
        "pos": int((df_out["label"] == 1).sum()),
        "neg": int((df_out["label"] == 0).sum()),
        "pos_ratio": float((df_out["label"] == 1).mean()),
        "parse_fail": int((df_out["parse_ok"] == 0).sum()),
        "truncated": int((df_out["ast_truncated"] == 1).sum()),
        "avg_nodes": float(df_out["ast_n_nodes"].mean()),
        "avg_edges": float(df_out["ast_n_edges"].mean()),
        "avg_error_nodes": float(df_out["ast_error_nodes"].mean()),
    },
}

summary_path = os.path.join(OUT_DIR, "summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("✓ Saved")
print("  CSV    :", csv_path)
print("  SUMMARY:", summary_path)

df_out[["id","label","ast_n_nodes","ast_error_nodes","parse_ok"]].head(5)


✓ Saved
  CSV    : qnn_dataset/qnn_dataset_v2\qnn_features_v2.csv
  SUMMARY: qnn_dataset/qnn_dataset_v2\summary.json


,id,label,ast_n_nodes,ast_error_nodes,parse_ok
0,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,1,551,0,1
1,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,0,551,0,1
2,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,0,461,0,1
3,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,0,17,0,1
4,VUL4J-1::src/main/java/com/alibaba/fastjson/se...,0,647,0,1


In [73]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Chọn số qubit bạn muốn
N_QUBITS = 16

feat_cols = [c for c in df_out.columns if c.startswith("feat_")]
X = df_out[feat_cols].values.astype(np.float32)
y = df_out["label"].values.astype(np.int64)

# Split theo vul_id để tránh leakage (rất quan trọng nếu fixed/vulnerable cùng CVE)
groups = df_out["vul_id"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx])
X_test = scaler.transform(X[test_idx])

pca = PCA(n_components=N_QUBITS, random_state=SEED)
Z_train = pca.fit_transform(X_train).astype(np.float32)
Z_test = pca.transform(X_test).astype(np.float32)

def to_angles(z: np.ndarray) -> np.ndarray:
    # clip theo quantile để tránh outlier làm saturate góc
    lo = np.quantile(z, 0.01, axis=0)
    hi = np.quantile(z, 0.99, axis=0)
    z = np.clip(z, lo, hi)
    z = (z - lo) / (hi - lo + 1e-8)  # [0,1]
    return (np.pi * z).astype(np.float32)  # [0, pi]

A_train = to_angles(Z_train)
A_test = to_angles(Z_test)

npz_path = os.path.join(OUT_DIR, f"qnn_ready_angles_{N_QUBITS}.npz")
np.savez_compressed(
    npz_path,
    X_train=A_train, y_train=y[train_idx],
    X_test=A_test, y_test=y[test_idx],
    train_ids=df_out.iloc[train_idx]["id"].values,
    test_ids=df_out.iloc[test_idx]["id"].values,
)

print("✓ Saved QNN-ready angles:", npz_path)
print("  A_train shape:", A_train.shape, "A_test shape:", A_test.shape)


✓ Saved QNN-ready angles: qnn_dataset/qnn_dataset_v2\qnn_ready_angles_16.npz
  A_train shape: (1218, 16) A_test shape: (360, 16)
